# Oryx research example

This notebook drives Oryx from plain Python through the **research host**, the `oryx` extension module built by `OryxPython`, rather than from inside Oasis.

**Setup (once):**
1. `uv run forge build all` builds `oryx.so` and writes the `.pth` file that makes `import oryx` work in `.venv`.
2. `uv sync` installs the `research` group (`ipykernel`, `numpy`, `pandas`).
3. Open this notebook in VS Code (or Jupyter) with the `.venv` interpreter as its kernel.

`oryx.init()` reads the nearest `oryx.yaml` above the working directory, here `Oasis/oryx.yaml`, and imports every script under its `scripting.roots` (`Oasis/scripts`: `nim.py`, `monte_carlo.py`). Their games and strategies register themselves as they are imported. Pass `oryx.init("path/to/oryx.yaml")` to use another file.

In [ ]:
import oryx

oryx.init()
oryx.__version__

## What is registered

The example scripts are already imported, so their classes can be used directly, next to the C++ strategies that ship with Oryx (which are named by string).

In [ ]:
from monte_carlo import MonteCarlo
from nim import Nim

print("games:     ", oryx.list_games())
print("strategies:", oryx.list_strategies())
oryx.describe_game(Nim)

## One match, step by step

A `Match` plays a game between strategies, one per seat. A game or strategy can be a registered class, a registry name, or a handle from `make_game`/`make_strategy`. `play()` runs the match to the end and returns each player's reward.

In [ ]:
match = oryx.Match(Nim, [MonteCarlo, "random"])
rewards = match.play()
print("rewards:", rewards)
print("moves:  ", match.history())

## Many matches

`simulate` runs a batch in C++ and returns a `BatchResult`. With `pandas` installed, `to_dataframe()` gives one row per player. A single strategy instead of a list plays every seat.

In [ ]:
result = oryx.simulate(Nim, [MonteCarlo, "random"], games=200, seed=7)
result.to_dataframe()

## A parameter sweep

Parameters are typed class fields (`playouts: int = 30`), so `make_strategy` can override them. How many playouts does Monte Carlo need to beat a random opponent reliably?

In [ ]:
import pandas as pd

rows = []
for playouts in (1, 2, 5, 10, 30):
    strategy = oryx.make_strategy(MonteCarlo, playouts=playouts)
    batch = oryx.simulate(Nim, [strategy, "random"], games=100, seed=1)
    rows.append({"playouts": playouts, "win_rate": batch.win_rates[0], "decisions": batch.decisions})

pd.DataFrame(rows).set_index("playouts")

## Writing a strategy in the notebook

A strategy only needs `decide(context)`. This one runs a negamax search over the lent state with `apply`/`undo`, then plays both opponents. Re-running the cell replaces the registration instead of raising, because the origin (this notebook) is the same. Try raising `depth` with `oryx.make_strategy("lookahead", depth=8)`.


In [ ]:
class Lookahead(oryx.Strategy, id="lookahead"):
    """Negamax for two-player zero-sum games, searched to a fixed depth; unfinished positions score 0."""

    depth: int = 4

    def decide(self, context):
        state = context.state
        return max(state.legal_actions(), key=lambda action: self.score(state, action, self.depth))

    def score(self, state, action, depth):
        mover = state.current_player()
        state.apply(action)
        if state.is_terminal():
            value = state.outcome()[mover]
        elif depth == 0:
            value = 0.0
        else:
            value = -max(self.score(state, reply, depth - 1) for reply in state.legal_actions())
        state.undo(action)
        return value


opponents = {"random": "random", "monte-carlo": MonteCarlo}
pd.DataFrame(
    {name: oryx.simulate(Nim, [Lookahead, opponent], games=100, seed=3).win_rates for name, opponent in opponents.items()},
    index=["lookahead", "opponent"],
)

## Timing and memory

`oryx.benchmark.benchmark` runs the same batch and also reports the time it took. With `memory=True` it counts the objects Oryx allocated during the run, exactly, through Oryx's own allocator (Python's allocations are not included).

In [ ]:
timing = oryx.benchmark.benchmark(Nim, [MonteCarlo, "random"], games=100, seed=1, memory=True)
print(f"{timing.matches_per_second:,.0f} matches/s, {timing.decisions_per_second:,.0f} decisions/s")
print(timing.memory)